In [ ]:
from dataclasses import dataclass
from langchain.messages import HumanMessage
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime

from dotenv import load_dotenv
load_dotenv()

# 定义context
# 方式一：dataclass注解
@dataclass
class UserContextV1:
    """用户上下文信息"""
    user_id: str = ""

# 方式二：typedict
class UserContextV2(TypedDict):
    """用户上下文信息"""
    user_id: str = ""

# 创建store
memory_store = InMemoryStore()

memory_store.put(
    ("preferences",),
    "user_001",
    {
        "hobby": "football"
    }
)

# 定义tools
@tool
def get_user_preference(runtime: ToolRuntime[UserContextV1]) -> str:
    """获取用户偏好"""
    user_id = runtime.context.user_id
    if user_id is None:
        return "not found"
    resp = runtime.store.get(
        ("preferences",),
        user_id
    )
    return resp.value

# 创建 agent
agent = create_agent(
    model="deepseek-chat",
    store=memory_store,
    tools=[get_user_preference],
    context_schema=UserContextV1
)

# 调用agent
resp = agent.invoke(
    {"messages": [HumanMessage("帮我查询user_001的爱好")]},
    context=UserContextV1("user_001")
)

for message in resp["messages"]:
    message.pretty_print()



ValidationError: 1 validation error for ChatDeepSeek
  Value error, If using default api base, DEEPSEEK_API_KEY must be set. [type=value_error, input_value={'model': 'deepseek-chat', 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error